# Pandas
Pandas is the tool you will use most in real AI/ML work... loading, cleaning, exploring, and reshaping tabular data.
This notebook covers the operations that appear in nearly every notebook.

A useful mental model: a DataFrame is a dictionary of NumPy arrays sharing a common index.
Once that clicks, most Pandas behavior... column selection, alignment during arithmetic, and even merge operations... 
stops feeling like memorized syntax and starts feeling logical.

## Series and DataFrame
A Series is a single labeled column (1D).
A DataFrame is a table of multiple Series sharing the same index (2D) - pandas core structure.

In [2]:
import pandas as pd

In [ ]:
s = pd.Series([10,20,30], name = 'score')
s

In [ ]:
df = pd.DataFrame({
    "name" : ["Asha", "Rohan", "Mira"],
    "score" : [85,92,78]
})

df

#AI/ML Use Case

Every CSV, SQL query result, or API response you load for ML becomes a DataFrame... it is the universal container for tabular ML data.

In [ ]:
# selecting columns

df["score"]     # single column -> Series

In [ ]:
df [["name","score"]]     # multiple columns -> DataFrame (note the double brackets)

In [ ]:
df.score        # attribute access -- works only if name has no spaces/special chars

## Reading & Inspecting Data

In [3]:
df = pd.read_csv("data.csv")

In [ ]:
df.head()

In [ ]:
df.tail(3)

In [ ]:
df.info()

In [ ]:
df.describe()   # gives maths of numerical data/columns

In [ ]:
df.shape

In [ ]:
df.columns

In [ ]:
df.dtypes

#AI/ML Use Case

This is always the first block of code in any EDA notebook, understand the shape and quality of data
before touching it

Interview Tip

Tip: df.info() and df.describe() together answer 'what columns exist, what types are they, and what does
the numeric distribution look like' in two lines.

## Selecting Data: loc & iloc

.loc selects by label/name. .iloc selects by integer position. This distinction is one of the most commonly
tested Pandas concepts.

In [ ]:
df.loc[0] # row with index label 0

In [ ]:
df.loc[0:2, "Row ID"] # rows 0-2 (inclusive), column "Row ID"

In [ ]:
df.loc[df["Index"] > 80] # conditional filtering

In [ ]:
df.iloc[0] # first row by position

In [ ]:
df.iloc[0:2, 0:1]    # first 2 rows, first column, by position

Common 
Mistake: .loc slicing is inclusive of the end label (0:2 includes index 2), while .iloc is exclusive like normal
Python slicing (0:2 excludes position 2).

## Filtering & Sorting 

In [ ]:
df[df["Index"] > 80] # filter rows

In [ ]:
df[(df["Index"] > 30) & (df["Index"] < 80)] # multiple conditions -- use & and |, with parentheses

In [ ]:
df.sort_values("Row ID", ascending=False)


In [ ]:
df.sort_values(["Index","Row ID"], ascending=[True, False])

Mistake: Using Python's and/or instead of & / | for combining row conditions on a DataFrame — and/or
do not work element-wise.

## value_counts(), unique(), nunique()

In [ ]:
df.columns

In [ ]:
df["Country"].value_counts()  #frequency of each country

In [ ]:
df["Country"].unique()    # array of distinct values

In [ ]:
df["Country"].nunique()     # number of distinct values

AI/ML Use Case: 
value_counts() is the fastest way to check class imbalance in a target column before
training a classifier.


## Missing Values 

Real datasets almost always have missing (NaN) values. Handling them correctly is a core EDA/cleaning
skill

In [ ]:
df.isnull().sum()         # missing values per column

In [ ]:
df.isnull().mean() * 100     # % missing per column

In [ ]:
df.fillna(0)       # fill with a const

In [ ]:
df["age"].fillna(df["age"].mean(), inplace=True) # fill with mean

In [ ]:
df.dropna() # drop rows with any missing value

In [ ]:
df.dropna(axis=1) # drop columns with any missing value

In [ ]:
df.dropna(thresh=5) # keep rows with at least 5 non-null values

AI/ML Use Case
Models cannot train on NaN values, every pipeline must explicitly impute or drop them before
modeling.

Common
Mistake: Dropping rows with dropna() by default without checking how much data is lost ,always
check df.isnull().sum() first.

## Duplicates

In [ ]:
df.duplicated().sum()  # count of duplicate rows

In [ ]:
df.drop_duplicates()   # remove duplicate rows

In [ ]:
df.drop_duplicates(subset=["Index"]) # based on specific column(s)

## apply(), map(), replace()

In [ ]:
df["score_pct"] = df["score"].apply(lambda x: x / 100)
df["grade"] = df["score"].map({90: "A", 80: "B", 70: "C"})
df["gender"] = df["gender"].replace({"M": "Male", "F": "Female"})
df["log_score"] = df["score"].apply(np.log)

AI/ML Use Case
apply() is used heavily for custom feature engineering; map()/replace() are used for label encoding and
value standardization.


## groupby

Splits data into groups based on a column, applies an aggregation, and combines the results — the 'splitapply-combine' pattern.

In [ ]:
df.groupby("city")["salary"].mean()
df.groupby("city").agg({"salary": "mean", "age": "max"})
df.groupby(["city","gender"])["salary"].mean()

AI/ML Use Case

Computing average target value per category, or aggregating transaction-level data into customer-level
features.

Interview Tip

Tip: groupby() alone returns a lazy GroupBy object, you always need to chain an aggregation like
.mean() or .agg() to see results.

## merge() & concat()

In [ ]:
# merge - SQL-style join on a key column
pd.merge(orders, customers, on="customer_id", how="left")
# how: "inner", "left", "right", "outer"


# concat - stack DataFrames
pd.concat([df1, df2], axis=0) # stack rows (same columns)
pd.concat([df1, df2], axis=1) # stack columns (same rows)

AI/ML Use Case: 

merge() combines tables from different sources (e.g. user info + transaction history)
into one ML-ready dataset.


## Basic Pivot Tables

In [ ]:
df.pivot_table(values="sales", index="region", columns="month", aggfunc="sum")

AI/ML Use Case:

Reshaping transaction data into a region-by-month sales matrix for trend analysis

## Datetime Handling

In [ ]:
df["date"] = pd.to_datetime(df["date"])
df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month
df["day_of_week"] = df["date"].dt.dayofweek
df.set_index("date").resample("M").mean() # monthly aggregation

Common
Mistake: Forgetting to convert a date column with pd.to_datetime() first — without it, .dt accessor
methods fail because the column stays a plain string.


##  Adding, Dropping & Renaming Columns

In [ ]:
df["bonus"] = df["score"] * 0.1 # add a new column
df.drop("bonus", axis=1) # drop a column (returns new df)
df.drop("bonus", axis=1, inplace=True) # drop in place
df.rename(columns={"score": "exam_score"})
df.rename(columns={"score": "exam_score"}, inplace=True)

AI/ML Use Case

Feature engineering is mostly this: deriving new columns from existing ones, and dropping columns that
leak information or add no value.

Mistake: Forgetting that most Pandas operations return a NEW DataFrame by default — without
inplace=True or reassignment, the original is unchanged.

## String Methods on a Series (.str accessor)
The .str accessor applies string methods to an entire text column at once, vectorized — no manual loop needed.

In [ ]:
df["Country"].str.lower()

In [ ]:
df["Country"].str.strip()

In [ ]:
df["Country"].str.contains("a")       # boolean mask

In [ ]:
df["email"].str.split("@").str[0]     # username part of an email

In [ ]:
df["Country"].str.len() # length of each string

AI/ML Use Case

Cleaning inconsistent text categories ("NY", " ny ", "New York") in one vectorized line instead of looping
row by row.

Always use .str methods instead of .apply(lambda x: x.lower()) when possible — .str is vectorized
and generally faster

## Method Chaining & query()


Real-world Pandas code is usually written as a chain of operations, which reads top-to-bottom like a
pipeline.

In [ ]:
result = (
 df[df["score"] > 50]
 .sort_values("score", ascending=False)
 .head(10)
)
# query() -- a readable alternative to boolean masking
df.query("score > 50 and age < 30")


AI/ML Use Case: 

Chained, readable preprocessing pipelines are the expected style in production
notebooks and code review.

## Multi-Index Basics
A DataFrame index made of multiple levels — the natural result of grouping by more than one column.

In [ ]:
grouped = df.groupby(["city", "gender"])["salary"].mean()
grouped.loc["Delhi"] # all genders within Delhi

In [ ]:
grouped.loc[("Delhi", "F")] # specific combination
grouped.reset_index() # flatten back into normal columns

AI/ML Use Case

reset_index() is used constantly after a groupby() to turn the result back into a flat, model-ready
DataFrame.

Common 
Mistake: Forgetting to call reset_index() after a groupby aggregation, then being confused why the
grouped column doesn't behave like a normal column.


## apply() Across Rows (axis=1)

In [ ]:
def classify(row):
    if row["score"] > 80 and row["attendance"] > 90:
     return "Top Performer"
     return "Regular"
    df["category"] = df.apply(classify, axis=1)

AI/ML Use Case: 
Creating a new feature that depends on multiple columns at once — axis=1 passes a full row into the
function instead of a single column.

Interview Tip

axis=1 with apply() is convenient but row-by-row in Python, so it's slower than vectorized operations
— fine for small/medium data, worth optimizing for very large data.

## Common Data-Cleaning Recipes

In [ ]:
# Standardize all column names
df.columns = df.columns.str.lower().str.strip().str.replace(" ", "_")

# Strip whitespace from every text column
for col in df.select_dtypes(include="object").columns:
 df[col] = df[col].str.strip()
    
# Convert a "looks numeric" column stuck as text
    df["price"] = pd.to_numeric(df["price"].str.replace(",", ""), errors="coerce")

# Standardize inconsistent category spelling
df["city"] = df["city"].str.lower().replace({
 "new york": "new_york", "ny": "new_york"
})

# Quick missing-value summary, sorted, percentage
(df.isnull().mean() * 100).sort_values(ascending=False).round(2)

AI/ML Use Case:

These five patterns alone resolve the majority of real-world messy-dataset problems
before any actual analysis begins.

## Practice Exercises


1. Load a CSV and print the names of all columns with more than 10% missing values.


In [ ]:
missing_pct = df.isnull().mean() * 100
print(missing_pct[missing_pct > 10].index.tolist())

2. Find the average value of a numeric column, grouped by a categorical column, sorted descending.

In [ ]:
df.groupby("category")["value"].mean().sort_values(ascending=False)

3. Create a new column that flags rows as "weekend" or "weekday" based on a date column.

In [ ]:
df["date"] = pd.to_datetime(df["date"])
df["day_type"] = df["date"].dt.dayofweek.apply(lambda x: "weekend" if x >= 5 else
"weekday")

4. Merge two DataFrames on a shared ID column, keeping only rows present in both.

In [ ]:
pd.merge(df1, df2, on="id", how="inner")